---
## 5. Interactive Web Map

The Folium map encodes **three** dimensions of the fire dataset at once:

| Visual variable | Attribute encoded | Design rationale |
|----------------|-------------------|------------------|
| **Marker colour** | Detection confidence | Traffic-light scheme: 🔵 low → 🟠 nominal → 🔴 high |
| **Marker radius** | Fire Radiative Power (FRP) | Proportional scaling: larger circle = more intense fire |
| **Layer (date picker)** | Acquisition date | Each day is a separate, toggleable layer |

### What's new in this version

The previous map drew every detection on a single static layer. This version adds a
**date picker**: each acquisition date becomes its own map layer, gathered under a
`GroupedLayerControl` panel. Because that panel is *non-exclusive*, several days can be
switched on at the same time and **compared directly** on one map — e.g. to see whether a
fire cluster is growing, shifting, or dying out from one day to the next.

Two complementary map products are produced:

1. **Date-picker map** — toggle days on/off for side-by-side comparison (Section 5.3).
2. **Animated time-slider map** — step or play through the days in sequence (Section 5.4).


In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import Fullscreen, MiniMap, GroupedLayerControl, TimestampedGeoJson
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)

print("Libraries imported.")


In [ ]:
from pathlib import Path

# ── Project paths ───────────────────────────────────────────────
# This works when the notebook is inside the "notebooks" folder.
NOTEBOOK_DIR = Path.cwd()

# If the current working directory is ".../notebooks", go one level up.
# Otherwise, keep the current directory as project root.
if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MAP_DIR = OUTPUT_DIR / "maps"

MAP_DIR.mkdir(parents=True, exist_ok=True)

# Two map products are saved by this notebook
MAP_OUTPUT          = MAP_DIR / "wildfire_interactive_map.html"   # date-picker map
MAP_OUTPUT_TIMESLIDER = MAP_DIR / "wildfire_timeslider_map.html"  # animated time-slider map

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Date-picker map output : {MAP_OUTPUT}")
print(f"Time-slider map output : {MAP_OUTPUT_TIMESLIDER}")


In [ ]:
# ── Load cleaned fire detections ─────────────────────────────────────

import pandas as pd
import geopandas as gpd

def _read_processed_fires():
    """
    Load the cleaned VIIRS FIRMS fire detections from data/processed/.
    """

    candidate_files = [
        PROCESSED_DIR / "firms_viirs_cleaned.gpkg",
        PROCESSED_DIR / "firms_viirs_cleaned.csv",
    ]

    existing = [path for path in candidate_files if path.exists()]

    if not existing:
        raise FileNotFoundError(
            "No cleaned fire dataset found in data/processed/. "
            "Expected one of: firms_viirs_cleaned.gpkg, firms_viirs_cleaned.csv"
        )

    path = existing[0]
    print(f"Loading processed fire detections from: {path.relative_to(PROJECT_ROOT)}")

    if path.suffix.lower() == ".gpkg":
        gdf = gpd.read_file(path)

    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

        if {"longitude", "latitude"}.issubset(df.columns):
            gdf = gpd.GeoDataFrame(
                df,
                geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
                crs="EPSG:4326"
            )
        else:
            raise ValueError(
                f"{path.name} does not contain longitude and latitude columns. "
                f"Available columns are: {list(df.columns)}"
            )

    else:
        raise ValueError(f"Unsupported file format: {path.suffix}")

    # Make sure the data is in WGS84 for web mapping
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    print(f"Loaded {len(gdf):,} fire detections.")
    print(f"Columns: {list(gdf.columns)}")

    return gdf


if "fires_gdf" not in globals():
    fires_gdf = _read_processed_fires()
else:
    print("Using existing fires_gdf from the current notebook session.")

fires_gdf.head()


In [ ]:
# ── Harmonise columns needed by the map ──────────────────────────────
required_columns = {"latitude", "longitude", "frp", "confidence"}
missing = required_columns.difference(fires_gdf.columns)
if missing:
    raise ValueError(f"The cleaned dataset is missing required columns: {sorted(missing)}")

fires_gdf["latitude"] = pd.to_numeric(fires_gdf["latitude"], errors="coerce")
fires_gdf["longitude"] = pd.to_numeric(fires_gdf["longitude"], errors="coerce")
fires_gdf["frp"] = pd.to_numeric(fires_gdf["frp"], errors="coerce")

fires_gdf = fires_gdf.dropna(subset=["latitude", "longitude", "frp"]).copy()
fires_gdf = fires_gdf[
    fires_gdf["latitude"].between(-90, 90)
    & fires_gdf["longitude"].between(-180, 180)
].copy()


def standardise_confidence(value):
    """Standardise FIRMS confidence values across text and numeric encodings."""
    if pd.isna(value):
        return "unknown"
    text = str(value).strip().lower()
    if text in {"h", "high"}:
        return "high"
    if text in {"n", "nominal", "medium", "moderate"}:
        return "nominal"
    if text in {"l", "low"}:
        return "low"
    if text.isdigit():
        number = int(text)
        if number >= 80:
            return "high"
        if number >= 30:
            return "nominal"
        return "low"
    return text


fires_gdf["confidence"] = fires_gdf["confidence"].apply(standardise_confidence)

if "acq_datetime" not in fires_gdf.columns:
    if {"acq_date", "acq_time"}.issubset(fires_gdf.columns):
        acq_time = fires_gdf["acq_time"].astype(str).str.zfill(4)
        fires_gdf["acq_datetime"] = pd.to_datetime(
            fires_gdf["acq_date"].astype(str) + " " + acq_time,
            format="%Y-%m-%d %H%M",
            errors="coerce",
            utc=True,
        )
    else:
        fires_gdf["acq_datetime"] = pd.NaT
else:
    fires_gdf["acq_datetime"] = pd.to_datetime(fires_gdf["acq_datetime"], errors="coerce", utc=True)

# ── Derive a clean per-day date string — this is the key driven by the date picker ────
# Every marker carries an 'acq_date' label (e.g. "2026-05-11"); the map builder
# turns each unique label into its own toggleable layer.
fires_gdf["acq_date"] = fires_gdf["acq_datetime"].dt.strftime("%Y-%m-%d")
fires_gdf["acq_date"] = fires_gdf["acq_date"].fillna("unknown date")

frp_max = fires_gdf["frp"].max()
fires_gdf["frp_normalised"] = fires_gdf["frp"] / frp_max if frp_max > 0 else 0.0

for col, default in {
    "bright_ti4": np.nan,
    "daynight": "",
    "continent": "not classified",
}.items():
    if col not in fires_gdf.columns:
        fires_gdf[col] = default

# Quick summary of the temporal coverage available to the date picker
date_counts = (
    fires_gdf.loc[fires_gdf["acq_date"] != "unknown date", "acq_date"]
    .value_counts()
    .sort_index()
)

print("Dataset prepared for mapping.")
print(f"Rows available for map: {len(fires_gdf):,}")
print(f"Confidence classes: {sorted(fires_gdf['confidence'].dropna().unique())}")
print(f"Acquisition dates available ({len(date_counts)}):")
for day, n in date_counts.items():
    print(f"  {day} : {n:,} detections")


---
## Map design

### Symbology

| Visual variable | Attribute encoded | Design rationale |
|---|---|---|
| **Marker colour** | Detection confidence | Traffic-light scheme: low, nominal, high |
| **Marker radius** | Fire Radiative Power (FRP) | Larger circles = more intense detections (square-root scaled) |
| **Map layer** | Acquisition date | One toggleable layer per day, driving the date picker |
| **Popup** | Full metadata | Date, FRP, confidence, brightness, continent, pass type, coordinates |

### The date picker

Each acquisition date is rendered as a separate `folium.FeatureGroup`. All of these
date layers are collected into a single **`GroupedLayerControl`** panel with
`exclusive_groups=False`, which renders them as **checkboxes** rather than radio
buttons. The practical effect:

- Tick **one** day → see that day in isolation.
- Tick **several** days → compare them on the same canvas (colours still distinguish confidence).
- Untick all but two consecutive days → a quick "before / after" view of fire progression.

By default the **two most recent days** load switched on so the comparison feature is
visible immediately; older days start unticked to keep the initial render light.

### Performance

To keep the saved HTML responsive, each day is independently capped at
`max_markers_per_date` detections, keeping that day's **highest-FRP** events. Capping
*per day* (rather than globally) ensures every day is fairly represented in a comparison,
instead of one extreme day crowding out the others.


In [ ]:
# ── Helper functions for map symbology ────────────────────────────────
CONFIDENCE_COLOUR_MAP = {
    "high": "#d62728",
    "nominal": "#ff7f0e",
    "low": "#1f77b4",
}
DEFAULT_COLOUR = "#888888"


def frp_to_radius(frp_value: float, min_radius: float = 3.0, max_radius: float = 18.0) -> float:
    """Map a normalised FRP value in [0, 1] to a Folium circle-marker radius."""
    sqrt_val = np.sqrt(np.clip(frp_value, 0, 1))
    return float(min_radius + sqrt_val * (max_radius - min_radius))


def safe_number(value, decimals=1, fallback="N/A"):
    """Format numeric values robustly for popup HTML."""
    number = pd.to_numeric(value, errors="coerce")
    if pd.isna(number):
        return fallback
    return f"{number:.{decimals}f}"


def build_popup_html(row: pd.Series) -> str:
    """Generate a compact HTML popup for one active fire detection."""
    confidence = str(row.get("confidence", "unknown")).lower()
    conf_colour = CONFIDENCE_COLOUR_MAP.get(confidence, DEFAULT_COLOUR)

    acq_dt = row.get("acq_datetime", None)
    dt_str = acq_dt.strftime("%Y-%m-%d %H:%M UTC") if pd.notna(acq_dt) else "Unknown"

    pass_code = str(row.get("daynight", "")).upper()
    daynight_label = "Daytime" if pass_code == "D" else "Nighttime" if pass_code == "N" else "Unknown"

    html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; min-width: 220px;">
      <b style="font-size:14px;">Fire detection</b><hr style="margin:4px 0;">
      <table style="border-collapse:collapse; width:100%;">
        <tr><td style="color:#555;">Date &amp; time</td><td><b>{dt_str}</b></td></tr>
        <tr><td style="color:#555;">FRP</td><td><b>{safe_number(row.get('frp'))} MW</b></td></tr>
        <tr><td style="color:#555;">Confidence</td><td><b style="color:{conf_colour};">{confidence.capitalize()}</b></td></tr>
        <tr><td style="color:#555;">Brightness T4</td><td>{safe_number(row.get('bright_ti4'))} K</td></tr>
        <tr><td style="color:#555;">Continent</td><td>{row.get('continent', 'not classified')}</td></tr>
        <tr><td style="color:#555;">Satellite pass</td><td>{daynight_label}</td></tr>
        <tr><td style="color:#555;">Coordinates</td><td>{safe_number(row.get('latitude'), 4)}°, {safe_number(row.get('longitude'), 4)}°</td></tr>
      </table>
    </div>
    """
    return html


def build_legend_html(with_datepicker_hint: bool = True) -> str:
    """Return the floating legend box. Optionally include a date-picker usage hint."""
    hint = ""
    if with_datepicker_hint:
        hint = (
            '<hr style="border-color:#555; margin:6px 0;">'
            '<b>Date picker</b><br>'
            '<small>Use the <i>Acquisition date</i> panel (top-right) to switch days '
            'on/off. Tick several days to compare them on one map.</small>'
        )
    return f"""
    <div style="
        position: fixed;
        bottom: 40px; left: 40px;
        z-index: 9999;
        background: rgba(20,20,20,0.88);
        color: #f0f0f0;
        padding: 14px 18px;
        border-radius: 8px;
        font-family: Arial, sans-serif;
        font-size: 13px;
        line-height: 1.8;
        border: 1px solid #555;
        max-width: 250px;
    ">
        <b style="font-size:14px;">Wildfire map legend</b><br>
        <hr style="border-color:#555; margin:6px 0;">
        <b>Colour → confidence</b><br>
        <span style="color:#d62728;">●</span> High confidence<br>
        <span style="color:#ff7f0e;">●</span> Nominal confidence<br>
        <span style="color:#1f77b4;">●</span> Low confidence<br>
        <span style="color:#888888;">●</span> Unknown / other<br>
        <hr style="border-color:#555; margin:6px 0;">
        <b>Size → Fire Radiative Power</b><br>
        ● Small = low FRP<br>
        ⬤ Large = high FRP<br>
        {hint}
        <hr style="border-color:#555; margin:6px 0;">
        <small>Source: NASA FIRMS VIIRS active fire data</small>
    </div>
    """


print("Map helper functions defined.")


In [ ]:
# ── Date-picker map builder ─────────────────────────────────────
def build_wildfire_map(
    gdf: gpd.GeoDataFrame,
    max_markers_per_date: int = 1_500,
    default_dates: int = 2,
) -> folium.Map:
    """
    Build an interactive Folium map with one toggleable layer per acquisition date.

    Each acquisition date becomes its own FeatureGroup. All date layers are
    collected under a single GroupedLayerControl (the "date picker"). Because
    that control is non-exclusive, multiple days can be switched on at once and
    compared directly on the same map. Within every layer, marker colour still
    encodes detection confidence and marker radius still encodes FRP.

    Parameters
    ----------
    gdf                  : Cleaned fire detections. Must contain the columns
                           'acq_date', 'frp', 'frp_normalised', 'confidence',
                           'latitude' and 'longitude'.
    max_markers_per_date : Cap on the number of markers drawn per day. The
                           highest-FRP detections of each day are kept. Capping
                           per day (not globally) keeps every day fairly
                           represented for comparison.
    default_dates        : How many of the most recent dates load switched on.

    Returns
    -------
    folium.Map
    """
    if gdf.empty:
        raise ValueError("The input GeoDataFrame is empty; no map can be built.")
    if "acq_date" not in gdf.columns:
        raise ValueError(
            "Expected an 'acq_date' column — run the column-harmonisation cell first."
        )

    dates = sorted(d for d in gdf["acq_date"].unique() if d != "unknown date")
    if not dates:
        raise ValueError("No valid acquisition dates were found in the dataset.")

    centre_lat = float(gdf["latitude"].median())
    centre_lon = float(gdf["longitude"].median())

    fire_map = folium.Map(
        location=[centre_lat, centre_lon],
        zoom_start=3,
        tiles=None,
        control_scale=True,
    )

    # Basemaps (switched via the small standard layer control) ---------------------
    folium.TileLayer("CartoDB dark_matter", name="Dark basemap").add_to(fire_map)
    folium.TileLayer("CartoDB positron", name="Light basemap").add_to(fire_map)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(fire_map)

    # One FeatureGroup per acquisition date ---------------------------------------
    recent_dates = set(dates[-default_dates:])
    date_layers = []

    for date_str in dates:
        day_mask = gdf["acq_date"] == date_str
        n_total = int(day_mask.sum())

        day_gdf = gdf[day_mask]
        if len(day_gdf) > max_markers_per_date:
            day_gdf = day_gdf.nlargest(max_markers_per_date, "frp")
            shown = f"top {max_markers_per_date:,} by FRP"
        else:
            shown = f"{n_total:,}"

        layer_label = f"{date_str}  ({n_total:,} detections)"
        day_layer = folium.FeatureGroup(name=layer_label, show=date_str in recent_dates)

        for _, row in day_gdf.iterrows():
            confidence_val = str(row.get("confidence", "unknown")).lower()
            fill_colour = CONFIDENCE_COLOUR_MAP.get(confidence_val, DEFAULT_COLOUR)
            radius = frp_to_radius(row.get("frp_normalised", 0.1))

            tooltip_text = (
                f"{date_str} | FRP: {safe_number(row.get('frp'), 0)} MW | "
                f"{confidence_val.capitalize()} confidence"
            )

            folium.CircleMarker(
                location=[row["latitude"], row["longitude"]],
                radius=radius,
                color=fill_colour,
                fill=True,
                fill_color=fill_colour,
                fill_opacity=0.65,
                weight=0.4,
                popup=folium.Popup(build_popup_html(row), max_width=290),
                tooltip=folium.Tooltip(tooltip_text),
            ).add_to(day_layer)

        day_layer.add_to(fire_map)
        date_layers.append(day_layer)
        print(f"  Layer built: {date_str}  ({shown} markers drawn)")

    # Standard control — only handles the basemaps. GroupedLayerControl sets
    # control=False on the date layers, so they do NOT appear here twice.
    folium.LayerControl(collapsed=True).add_to(fire_map)

    # The date picker: a non-exclusive grouped control => checkboxes => compare days.
    GroupedLayerControl(
        groups={"Acquisition date (UTC)": date_layers},
        exclusive_groups=False,
        collapsed=False,
    ).add_to(fire_map)

    MiniMap(toggle_display=True).add_to(fire_map)
    Fullscreen().add_to(fire_map)

    fire_map.get_root().html.add_child(
        folium.Element(build_legend_html(with_datepicker_hint=True))
    )

    return fire_map


print("Date-picker map builder defined.")


In [ ]:
# ── Build, save and display the date-picker map ────────────────────────
print("Building interactive wildfire map (date picker)...")
wildfire_map = build_wildfire_map(
    fires_gdf,
    max_markers_per_date=1_500,
    default_dates=2,
)

wildfire_map.save(str(MAP_OUTPUT))
print(f"\nMap saved to: {MAP_OUTPUT.relative_to(PROJECT_ROOT)}")
print("Tip: open the 'Acquisition date (UTC)' panel (top-right) and tick "
      "two or more days to compare them.")

display(wildfire_map)


---
## 5.4 Alternative view — animated time slider

The date-picker map above is best for **comparing** days (several layers visible at
once). For seeing the dataset evolve **in sequence**, a time slider is more natural.

This second map uses `folium.plugins.TimestampedGeoJson`: every detection carries its
`acq_date` as a timestamp, and Folium renders a slider with play / step / loop controls
along the bottom. With `duration="P1D"` each detection is shown only on its own day, so
dragging the slider gives a clean day-by-day animation of global fire activity rather
than an ever-accumulating pile of points.

Both maps are built from the **same** `fires_gdf` and the **same** symbology helpers —
they are just two different ways of exposing the time dimension.


In [ ]:
# ── Animated time-slider map builder ───────────────────────────────
def build_timeslider_map(gdf: gpd.GeoDataFrame, max_markers: int = 3_000) -> folium.Map:
    """
    Build a Folium map with an animated time slider stepping through acquisition days.

    Detections are encoded as GeoJSON features whose 'time' property is the
    acquisition date. TimestampedGeoJson then renders play / step / loop
    controls. Colour encodes confidence and radius encodes FRP, exactly as in
    the date-picker map.

    Parameters
    ----------
    gdf         : Cleaned fire detections (needs 'acq_datetime', 'frp',
                  'frp_normalised', 'confidence', 'latitude', 'longitude').
    max_markers : Global cap on the number of detections animated, keeping the
                  highest-FRP events (the slider HTML embeds every point, so a
                  cap keeps the file responsive in a browser).

    Returns
    -------
    folium.Map
    """
    if gdf.empty:
        raise ValueError("The input GeoDataFrame is empty; no map can be built.")

    timed = gdf[gdf["acq_datetime"].notna()].copy()
    if timed.empty:
        raise ValueError("No detections have a valid 'acq_datetime' to animate.")

    if len(timed) > max_markers:
        timed = timed.nlargest(max_markers, "frp")
        print(f"Animation clipped to top {max_markers:,} detections by FRP.")

    features = []
    for _, row in timed.iterrows():
        confidence_val = str(row.get("confidence", "unknown")).lower()
        colour = CONFIDENCE_COLOUR_MAP.get(confidence_val, DEFAULT_COLOUR)
        features.append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [float(row["longitude"]), float(row["latitude"])],
            },
            "properties": {
                "time": row["acq_datetime"].strftime("%Y-%m-%d"),
                "popup": build_popup_html(row),
                "tooltip": f"FRP: {safe_number(row.get('frp'), 0)} MW | {confidence_val.capitalize()}",
                "icon": "circle",
                "iconstyle": {
                    "fillColor": colour,
                    "color": colour,
                    "fillOpacity": 0.7,
                    "stroke": False,
                    "radius": frp_to_radius(row.get("frp_normalised", 0.1)),
                },
            },
        })

    centre_lat = float(timed["latitude"].median())
    centre_lon = float(timed["longitude"].median())

    ts_map = folium.Map(
        location=[centre_lat, centre_lon],
        zoom_start=3,
        tiles="CartoDB dark_matter",
        control_scale=True,
    )

    TimestampedGeoJson(
        {"type": "FeatureCollection", "features": features},
        period="P1D",                  # slider advances one day at a time
        duration="P1D",                # each point is visible only on its own day
        transition_time=400,
        add_last_point=False,
        auto_play=False,
        loop=False,
        loop_button=True,
        max_speed=3,
        date_options="YYYY-MM-DD",
        time_slider_drag_update=True,
    ).add_to(ts_map)

    Fullscreen().add_to(ts_map)
    ts_map.get_root().html.add_child(
        folium.Element(build_legend_html(with_datepicker_hint=False))
    )

    return ts_map


print("Time-slider map builder defined.")


In [ ]:
# ── Build, save and display the animated time-slider map ──────────────────
print("Building animated time-slider wildfire map...")
timeslider_map = build_timeslider_map(fires_gdf, max_markers=3_000)

timeslider_map.save(str(MAP_OUTPUT_TIMESLIDER))
print(f"\nMap saved to: {MAP_OUTPUT_TIMESLIDER.relative_to(PROJECT_ROOT)}")
print("Tip: press play on the slider at the bottom, or drag it to step through days.")

display(timeslider_map)


---
## Summary and reflection

### Workflow summary

| Step | Action | Main output |
|---|---|---|
| 1 | Retrieved recent active fire detections from NASA FIRMS | Raw CSV in `data/raw/` |
| 2 | Cleaned attributes, parsed acquisition time and created point geometries | Cleaned file in `data/processed/` |
| 3 | Explored FRP, confidence, day/night patterns and spatial distribution | Figures in `outputs/figures/` |
| 4 | Built an interactive **date-picker** web map (toggle/compare days) | `outputs/maps/wildfire_interactive_map.html` |
| 5 | Built an animated **time-slider** web map (day-by-day playback) | `outputs/maps/wildfire_timeslider_map.html` |

### Interpretation

The final maps translate the cleaned active-fire dataset into two complementary
exploratory tools. Marker colour communicates detection confidence and marker size
communicates fire radiative power, while the **time dimension** is now explicit: the
date-picker map exposes each day as a toggleable layer for direct side-by-side
comparison, and the time-slider map animates the same detections in sequence. Together
they make it possible not just to see *where* intense fires are, but *how the pattern
shifts from one day to the next*.

### Limitations

- The maps show satellite fire detections, not complete burned-area perimeters.
- Cloud cover, smoke and sensor overpass timing can create detection gaps, so an empty
  day in the date picker does not necessarily mean an absence of fire.
- Very large days are clipped to their strongest-FRP detections for browser performance.
- Low-confidence detections may include false positives from industrial heat sources.
- The retrieval window is short (a few days), so the date picker compares days, not seasons.

### Possible extensions

- Add national boundaries or protected-area polygons and perform spatial joins.
- Add a second grouped control for day/night pass alongside the date picker.
- Compare VIIRS detections with MODIS active fire products on parallel layers.
- Pull a longer time window and aggregate to weekly layers for seasonal comparison.



### Key Findings

1. **Q1 (Severity):** The highest FRP values cluster in tropical and subtropical regions, consistent with the literature on savanna and deforestation fires. The most extreme single detections often exceed several thousand MW.

2. **Q2 (Intensity by continent):** Median FRP varies substantially across continents. The heavy right-skewed distributions on a log scale highlight that the "average" wildfire is orders of magnitude less intense than the largest events — typical summary statistics can be very misleading without appropriate visualisation.

3. **Q3 (Day vs. Night):** The global split is approximately 50/50, reflecting the satellite's polar orbit design. Regional deviations provide information about local detection conditions and fire regimes.

4. **Q4 (Confidence vs. FRP):** If median FRP increases monotonically from low → nominal → high confidence, this validates using confidence as a map symbolization variable. It also confirms that the sensor's internal confidence algorithm is thermally consistent.

### Limitations & Future Work

- **Detection gaps:** Cloud cover and smoke plumes can obscure active fires from optical and thermal sensors, causing systematic underdetection in regions with convective cloud cover.
- **Temporal sampling:** With only 1–2 VIIRS overpasses per day at mid-latitudes, fires that ignite and extinguish between passes are missed entirely.
- **False detections:** Industrial heat sources (smelters, gas flares) can produce false positives, especially in lower confidence classes.
- **Extensions:** Integrating MODIS Active Fire data alongside VIIRS would provide multi-sensor corroboration. Adding country boundaries and protected area polygons via a spatial join would allow policy-relevant analysis of fires in national parks or buffer zones.


---
## Reproducibility Checklist

Before submission, confirm all boxes can be ticked:

- [ ] Notebook runs **top to bottom** without errors after `Kernel → Restart & Run All`
- [ ] All file paths are **relative** (no `/Users/...` or `C:\\...` absolute paths)
- [ ] `data/raw/` is listed in `.gitignore` (large CSV files not committed to Git)
- [ ] `environment.yml` is present in the repository root and lists `folium`
- [ ] `README.md` contains: project description, data source link, setup instructions, execution order
- [ ] `outputs/maps/` contains both exported HTML maps (date-picker and time-slider)

---
*Notebook generated for SDS210 Project 2 · University of Zurich · FS 2025/26*
